In [ ]:
import pandas as pd
import numpy as np


El diseño del software debe reflejar la estricta separación física de los componentes de
hardware.
• Herencia y Clases: Definir entidades lógicas independientes para ControladorPID,
PlantaMotor y Sensor.
• Encapsulamiento del Estado Físico: La clase PlantaMotor debe modelar la inercia
del rotor. Su velocidad actual debe ser un atributo privado. Un programador externo no
debe poder alterar la velocidad de 0 a 1000 RPM de forma instantánea; el motor solo
debe recibir una señal de acción (voltaje) y actualizar su velocidad progresivamente en
el tiempo mediante métodos internos.
• Entregable: Diagrama de clases UML que muestre las relaciones de composición
(ejemplo: un SistemaLazoCerrado que contiene un motor, un sensor y un controlador),
junto con la inicialización del repositorio Git/GitHub y el esqueleto de las clases

In [1]:
class PlantaMotor:
    """
    Representa el componente físico del motor, incluyendo la inercia de rotor

    Atributos:
    inercia (float): Constante de tiempo (RPM)
    kv (float): Ganancia (RPM/V)
    v (float): Velocidad actual (RPM)
    """
    def __init__(self, inercia: float, kv: float):
        self.inercia = inercia
        self.ganancia = kv
        v = 0.0
        self.__vel_act: float = v

    def get_vel(self) -> float:
        # Devuelve la velocidad actual en (RPM)
        return self.__vel_act

    def aplicar_Vol(self, V: float, dt: float):
        """
        Actualiza la velocidad de forma progresiva según la ecuación de inercia
        """
        velocidad_objetivo = V * self.ganancia 

        aceleracion = (velocidad_objetivo - self.__vel_act) / self.inercia

        self.__vel_act += aceleracion * dt

In [2]:
class Sensor:
    """
    Mide velocidad en RPM.

    Args:
    id_sensor (str): Identificacion sensor
    vel (float): Velocidad medida (RPM)
    """
    def __init__(self, id_sensor: str, vel: float):
        self.id_sensor = id_sensor
        self.vel = vel

    def medir_vel(self) -> float:
        return self.vel

In [3]:
class ControladorPID:
    """
    Controlador que regula voltaje para reducir error
    
    Args:
    kp (float): Control Proporcional
    ki (float): Control Integral
    kd (float): Control Derivativo
    """

    def __init__(self, kp: float, ki: float, kd: float):
        self.proporcional = kp
        self.integral = ki
        self.derivativo = kd

        self.error_int = 0.0
        self.error_prev = 0.0

    def calcular_voltaje(self, setpoint: float, lectura: float, dt: float) -> float:
        error = setpoint - lectura

        # 1. Proporcional: reacciona al error actual
        P = self.proporcional * error

        # 2. Integral: acumula el error acumulado en el tiempo
        self.error_int += error * dt
        I = self.integral * self.error_int

        # 3. Derivativo: reacciona a la velocidad de cambio del error
        derivada = (error - self.error_prev) / dt
        D = self.derivativo * derivada

        self.error_prev = error
        
        return P + I + D
